In [ ]:
from pyspark.sql import functions as F

CONTROL_TABLE = "ctl_monthly_load"
BRONZE_TABLE = "bronze_ontime"
AUDIT_COLS = {"_period_key", "_source_file", "_ingested_at"}

control = spark.table(CONTROL_TABLE)
bronze = spark.table(BRONZE_TABLE)
source_cols = [c for c in bronze.columns if c not in AUDIT_COLS]

print("=== 1. Control table ===")
for r in control.groupBy("download_status", "load_status").count().orderBy("download_status").collect():
    print(f"{r['download_status']} / {r['load_status']}: {r['count']}")

print("\n=== 2. Coverage ===")
expected = {r.period_key for r in control.select("period_key").collect()}
actual = {r._period_key for r in bronze.select("_period_key").distinct().collect()}
print(f"Expected months: {len(expected)}")
print(f"Months in Bronze: {len(actual)}")
print(f"Missing from Bronze: {sorted(expected - actual) or 'none'}")
print(f"Unexpected in Bronze: {sorted(actual - expected) or 'none'}")

monthly = (
    bronze.groupBy("_period_key")
    .agg(
        F.count("*").alias("flights"),
        F.countDistinct("year", "month").alias("label_combos"),
        F.min(F.concat_ws("-", F.col("year"), F.lpad(F.col("month"), 2, "0"))).alias("label"),
    )
    .cache()
)

print("\n=== 3. Flights by year ===")
total = 0
for r in (
    monthly.withColumn("yr", F.substring("_period_key", 1, 4))
    .groupBy("yr").agg(F.sum("flights").alias("flights"), F.count("*").alias("months"))
    .orderBy("yr").collect()
):
    total += r["flights"]
    print(f"{r['yr']}: {r['flights']:,} flights ({r['months']} months)")
print(f"Total: {total:,} flights")

print("\n=== 4. Smallest and largest months ===")
for label, order in [("Smallest", F.asc("flights")), ("Largest", F.desc("flights"))]:
    rows = monthly.orderBy(order).limit(3).collect()
    print(f"{label}: " + ", ".join(f"{r['_period_key']} ({r['flights']:,})" for r in rows))

print("\n=== 5. Label check ===")
mismatched = monthly.filter(
    (F.col("label_combos") != 1) | (F.col("label") != F.col("_period_key"))
).select("_period_key", "label", "label_combos").collect()
print(f"Months where year/month columns don't match the partition: {[(r[0], r[1], r[2]) for r in mismatched] or 'none'}")

print("\n=== 6. Column consistency ===")
non_null = bronze.groupBy("_period_key").agg(*[F.count(F.col(c)).alias(c) for c in source_cols]).collect()
empty_counts = {r["_period_key"]: sum(1 for c in source_cols if r[c] == 0) for r in non_null}
print(f"Source columns: {len(source_cols)}")
print(f"Fully empty columns per month: min {min(empty_counts.values())}, max {max(empty_counts.values())}")
always_empty = [c for c in source_cols if all(r[c] == 0 for r in non_null)]
print(f"Columns empty in every month ({len(always_empty)}): {always_empty}")

=== 1. Control table ===
downloaded / loaded: 90

=== 2. Coverage ===
Expected months: 90
Months in Bronze: 90
Missing from Bronze: none
Unexpected in Bronze: none

=== 3. Flights by year ===
2019: 7,422,037 flights (12 months)
2020: 4,688,354 flights (12 months)
2021: 5,995,397 flights (12 months)
2022: 6,729,125 flights (12 months)
2023: 6,847,899 flights (12 months)
2024: 7,079,061 flights (12 months)
2025: 7,001,619 flights (12 months)
2026: 3,488,373 flights (6 months)
Total: 49,251,865 flights

=== 4. Smallest and largest months ===
Smallest: 2020-05 (180,617), 2020-06 (223,732), 2020-04 (313,382)
Largest: 2019-07 (659,029), 2019-08 (658,461), 2020-03 (648,229)

=== 5. Label check ===
Months where year/month columns don't match the partition: none

=== 6. Column consistency ===
Source columns: 109
Fully empty columns per month: min 16, max 32
Columns empty in every month (16): ['div4_airport', 'div4_airport_id', 'div4_airport_seq_id', 'div4_wheels_on', 'div4_total_g_time', 'div4_